In [ ]:
####################################################################################
# Starter code supplied by the Datacamp challenge – kept as-is for reproducibility #
####################################################################################

# Importing libraries
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.regression import RandomForestRegressor
from pyspark.sql.functions import col, dayofmonth, month, year,  to_date, to_timestamp, weekofyear, dayofweek
from pyspark.ml.feature import StringIndexer
from pyspark.ml.evaluation import RegressionEvaluator

# Initialize Spark session
my_spark = SparkSession.builder.appName("SalesForecast").getOrCreate()

# Importing sales data
sales_data = my_spark.read.csv(
    "Online Retail.csv", header=True, inferSchema=True, sep=",")

# Convert InvoiceDate to datetime 
sales_data = sales_data.withColumn("InvoiceDate", to_date(
    to_timestamp(col("InvoiceDate"), "d/M/yyyy H:mm")))

In [ ]:
#################################################################
# 1.  DATA CLEANING  –  remove returns, missing keys, negatives #
#################################################################
# → 541 k rows  →  406 k rows after cleaning

clean = (sales_data
         .filter("Quantity > 0 AND UnitPrice > 0 AND CustomerID IS NOT NULL")
         .select("Country", "StockCode", "InvoiceDate", "Quantity"))




In [ ]:
#################################
# 2.  AGGREGATE TO DAILY LEVEL  #
#################################

# Assignment requires **one row per Country*StockCode*day**
# We also cast Quantity → double to avoid any int+str surprises

daily = (clean
         .withColumn("Quantity", col("Quantity").cast("double"))
         .groupBy("Country", "StockCode", "InvoiceDate")
         .agg({"Quantity": "sum"})
         .withColumnRenamed("sum(Quantity)", "Quantity"))

In [ ]:
###########################
# 3.  TRAIN / TEST SPLIT  #
###########################

split_date = "2011-09-25"
train_df = daily.filter(col("InvoiceDate") <= split_date)
test_df  = daily.filter(col("InvoiceDate") >  split_date)

pd_daily_train_data = train_df.toPandas() # Assignment requires deliverable n1 to be pandas

In [ ]:
###########################################
# 4.  FEATURE ENGINEERING - 1st attempt #
###########################################

# Dead end : kept original granularity → 2 M rows → OOM
# Dead end : used **all** columns which lead to categorical explosion

# ❶  daily_sales_v1 = sales_raw.groupBy(
#        "Country","StockCode","InvoiceDate","Year","Month","Day","Week","DayOfWeek"
#    ).agg({"Quantity":"sum","UnitPrice":"avg"})
# ❷  feat_cols = ["CountryIndex","StockCodeIndex","Month","Year","DayOfWeek","Day","Week"]
# ❸  rf = RandomForestRegressor(maxBins=4000)   # blew JVM heap

In [ ]:
######################################################
# 5. FEATURE ENGINEERING -  2nd attempt w/ pipeline  #
######################################################

# Strip to **daily** grain only
# Add minimal calendar features
# Force indexed columns to **numeric** so tree uses 1 bin per feature

from pyspark.ml import Transformer
from pyspark.ml.param.shared import HasInputCol, HasOutputCol
from pyspark import keyword_only

class NumericCast(Transformer, HasInputCol, HasOutputCol):
    @keyword_only
    def __init__(self, inputCol=None, outputCol=None):
        super(NumericCast, self).__init__()
        kwargs = self._input_kwargs
        self.setParams(**kwargs)
    @keyword_only
    def setParams(self, **kwargs):
        return self._set(**kwargs)
    def _transform(self, df):
        return df.withColumn(self.getOutputCol(),
                             col(self.getInputCol()).cast("int"))

calendar = (train_df
            .withColumn("month", month("InvoiceDate"))
            .withColumn("dow", dayofweek("InvoiceDate"))
            .withColumn("woy", weekofyear("InvoiceDate")))

country_idx = StringIndexer(inputCol="Country", outputCol="Country_tmp", handleInvalid="keep")
stock_idx   = StringIndexer(inputCol="StockCode", outputCol="StockCode_tmp", handleInvalid="keep")

country_num = NumericCast(inputCol="Country_tmp", outputCol="CountryIndex")
stock_num   = NumericCast(inputCol="StockCode_tmp", outputCol="StockCodeIndex")

assembler = VectorAssembler(inputCols=["CountryIndex","StockCodeIndex","month","dow","woy"],
                            outputCol="features", handleInvalid="keep")

rf = RandomForestRegressor(labelCol="Quantity", featuresCol="features",
                           numTrees=50, maxDepth=5, maxBins=32, seed=42)

pipeline = Pipeline(stages=[country_idx, stock_idx, country_num, stock_num, assembler, rf])
model    = pipeline.fit(calendar)

In [ ]:
##############################################
# 6.  MODEL EVALUATION - MAE on daily level  #
##############################################
test_cal = (test_df
            .withColumn("month", month("InvoiceDate"))
            .withColumn("dow", dayofweek("InvoiceDate"))
            .withColumn("woy", weekofyear("InvoiceDate")))

pred = model.transform(test_cal)
mae  = float(RegressionEvaluator(labelCol="Quantity", predictionCol="prediction",
                                 metricName="mae").evaluate(pred))
print("MAE:", mae)   # assignment deliverable n2

In [ ]:
################################
# 7.  FORECASTING FOR WEEK 39  #
################################

# Dead end: filtered on test_df (history)
# Corrected approach: build every (Country, StockCode) * (week-39 days)

universe = clean.select("Country", "StockCode").distinct()

days = my_spark.createDataFrame(
    [(d,) for d in ["2011-09-26", "2011-09-27", "2011-09-28",
                    "2011-09-29", "2011-09-30", "2011-10-01", "2011-10-02"]],
    ["InvoiceDate"]
).select(to_date("InvoiceDate", "yyyy-MM-dd").alias("InvoiceDate"))

future = (universe.crossJoin(days)
          .withColumn("month", month("InvoiceDate"))
          .withColumn("dow", dayofweek("InvoiceDate"))
          .withColumn("woy", weekofyear("InvoiceDate")))

quantity_sold_w39 = int(
    model.transform(future).agg({"prediction": "sum"}).collect()[0][0]
)                                           # deliverable 3
quantity_sold_w39                         # last line – auto-grader picks it up

In [ ]:
#######################
# 8.  CLEAN SHUTDOWN  #
#######################
# An optional best practice
my_spark.stop()